# Lekce 10 - AI agenti v produkci

V této lekci se naučíte **produkční vzory** pro AI agenty pomocí Microsoft Agent Framework (Python). Pokrýváme:

- **Pozorovatelnost** — přidávání časování a logování interakcí agenta
- **Hodnocení** — použití hodnotícího agenta pro skórování kvality odpovědí
- **Řízení nákladů** — strategie pro optimalizaci tokenů a výběr modelu

Scénář je **cestovní agent**, který pomáhá uživatelům plánovat cesty, s monitorováním a hodnocením navrchu.


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produkční úvahy

Přechod AI agentů z prototypů do produkce vyžaduje pečlivou pozornost třem pilířům:

1. **Pozorovatelnost** — Potřebujete viditelnost do toho, co agent dělá, jak dlouho to trvá a jaké nástroje volá. Bez trasování a logování je ladění problémů v produkci téměř nemožné.

2. **Hodnocení** — Automatizované kontrolní mechanizmy kvality zajišťují, že odpovědi agenta zůstávají přesné, úplné a užitečné v průběhu času. Agent hodnotitel může skórovat odpovědi podle definovaných kritérií.

3. **Řízení nákladů** — Použití tokenů přímo ovlivňuje náklady. Strategie jako optimalizace promptu, výběr modelu a kešování pomáhají udržet výdaje pod kontrolou bez ztráty kvality.


## Vytváření pozorovatelného agenta

Definujeme cestovní nástroje a zabalíme volání agenta s měřením času, abychom mohli sledovat latenci. Ve výrobě byste integrovali OpenTelemetry nebo podobný sledovací backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzory hodnocení

Běžným produkčním vzorem je použití druhého agenta jako **hodnotitele**. Hodnotitel hodnotí odpověď primárního agenta podle předem definovaných kritérií, jako je úplnost, správnost a užitečnost.

To umožňuje:
- Automatizované kontrolní brány kvality před tím, než odpovědi dorazí k uživatelům
- Zachycení regrese při změnách v promptu nebo modelech
- Nepřetržité sledování výkonu agenta v čase


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie řízení nákladů

Kontrola nákladů je klíčová pro produkční AI agenty. Zde jsou hlavní strategie:

| Strategie | Popis |
|---|---|
| **Optimalizace promptu** | Udržujte systémové instrukce stručné. Odstraňte nadbytečný kontext, aby se snížil počet vstupních tokenů. |
    "| **Výběr modelu** | Používejte menší, levnější modely (např. GPT-4.1-mini) pro jednoduché úkoly jako klasifikace nebo extrakce a vyhraďte větší modely pro složité uvažování. |\n",
| **Cacheování** | Ukládejte do mezipaměti výsledky nástrojů a časté dotazy, abyste zabránili nadbytečným voláním API. |
| **Tokenové limity** | Nastavte limity `max_tokens`, abyste zabránili nečekaně dlouhým odpovědím. |
| **Sdružování** | Tam, kde je to možné, seskupte více uživatelských dotazů do jednoho volání API. |

V praxi funguje dobře vícestupňový přístup: jednoduché požadavky směrujte na rychlý, levný model a složité dotazy eskalujte pouze na schopnější model.


## Shrnutí

V této lekci jste se naučili:

1. **Přidat sledovatelnost** k interakcím agentů pomocí časování a protokolování, čímž se položily základy pro sledování a monitorování.
2. **Automaticky vyhodnocovat odpovědi agentů** pomocí hodnotícího agenta, který boduje úplnost, přesnost a užitečnost.
3. **Řídit náklady** prostřednictvím optimalizace promptů, výběru modelu, cachování a rozpočtů tokenů.

Tyto produkční vzory pomáhají zajistit, aby vaši AI agenti byli spolehliví, měřitelní a nákladově efektivní ve velkém měřítku.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
